# STAT3009 — Midterm workflow rehearsal (solution)

**Instructor version.** This notebook is fully executed. The student version is
`STAT3009_Midterm_Rehearsal.ipynb`.

**Suggested time:** 35–40 minutes

This rehearsal uses a separate mini dataset with local test labels. The real
midterm hides test labels and uses `submit()`.


## 1. Check the data contract (3 min)

`raw_ratings.csv` has `user, item, rating`; `test_pairs.csv` has `user, item`;
`test_labels.csv` contains local test ratings.

Look at the tables and answer aloud:

1. How many observed ratings and test pairs are there?
2. Why can `U71371` not be used directly as a NumPy array index?
3. What must be true about the user and item vocabularies before fitting?

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 3009
for candidate in [Path('.'), Path('rehearsal'), Path('nb/rehearsal')]:
    if (candidate / 'raw_ratings.csv').exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError('Open this notebook from the rehearsal folder.')
sys.path.insert(0, str(DATA_DIR.resolve()))

from TabRS import GlobalMeanRS, UserMeanRS, ItemMeanRS, BiasedMF

np.random.seed(RANDOM_STATE)

train = pd.read_csv(DATA_DIR / 'raw_ratings.csv')
test = pd.read_csv(DATA_DIR / 'test_pairs.csv')
test_labels = pd.read_csv(DATA_DIR / 'test_labels.csv')['rating'].to_numpy(dtype=float)

print('Observed ratings:', len(train))
print('Test pairs:', len(test))
display(train.head())
display(test.head())

Observed ratings: 1598
Test pairs: 402


,user,item,rating
0,U71371,I55111,3.630381
1,U13437,I83182,5.000000
2,U97950,I85222,3.147805
3,U17603,I73037,2.315970
4,U22082,I98443,3.304730


,user,item
0,U23619,I33461
1,U22795,I83182
2,U82852,I16859
3,U56777,I45464
4,U93731,I85270


## 2. Package the preprocessing (8 min)

Write **one function** that returns the arrays and vocabulary sizes:

```python
X_train, X_test, y_train, n_users, n_items = encode_ratings(train, test)
```

Fit each `LabelEncoder` on the **combined train and test IDs**, transform both
tables, preserve row order, and return integer NumPy arrays. This is the same
contract used by the supplied models.

In [2]:
def encode_ratings(train, test):
    user_encoder = LabelEncoder().fit(pd.concat([train['user'], test['user']]))
    item_encoder = LabelEncoder().fit(pd.concat([train['item'], test['item']]))
    train_users = user_encoder.transform(train['user'])
    test_users = user_encoder.transform(test['user'])
    train_items = item_encoder.transform(train['item'])
    test_items = item_encoder.transform(test['item'])
    X_train = np.column_stack([train_users, train_items]).astype(int)
    X_test = np.column_stack([test_users, test_items]).astype(int)
    y_train = train['rating'].to_numpy(dtype=float)
    return X_train, X_test, y_train, len(user_encoder.classes_), len(item_encoder.classes_)

X_train, X_test, y_train, n_users, n_items = encode_ratings(train, test)
print('X_train:', X_train.shape, '| y_train:', y_train.shape, '| X_test:', X_test.shape)
print('Users:', n_users, '| Items:', n_items)

X_train: (1598, 2) | y_train: (1598,) | X_test: (402, 2)
Users: 80 | Items: 40


## 3. Baseline scoreboard (7 min)

Complete the helper `fit_and_score`. It should fit a supplied model on
`X_train, y_train`, predict `X_test`, and return local RMSE.

Then use it to score the three baselines. Which one wins on this mini dataset?

In [3]:
def fit_and_score(model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return root_mean_squared_error(test_labels, pred)

baselines = {
    'Global mean': GlobalMeanRS(n_users, n_items),
    'User mean': UserMeanRS(n_users, n_items),
    'Item mean': ItemMeanRS(n_users, n_items),
}
rows = []
for name, model in baselines.items():
    rows.append({'method': name, 'RMSE': fit_and_score(model)})
display(pd.DataFrame(rows).sort_values('RMSE').round(3))

,method,RMSE
0,Global mean,0.939
2,Item mean,0.951
1,User mean,0.972


## 4. Fit the starting MF model (5 min)

The supplied `BiasedMF` predicts

$$\widehat r_{ui}=\mu+a_u+b_i+p_u^\mathsf{T}q_i.$$

Fit the starting model below, predict `X_test`, and score it. This model is
**not** tuned yet.

In [4]:
untuned = BiasedMF(
    n_users=n_users, n_items=n_items,
    K=2, lambda_=3.0, n_iters=3, random_state=RANDOM_STATE,
)
untuned.fit(X_train, y_train)
untuned_pred = untuned.predict(X_test)
print('starting MF RMSE:', round(root_mean_squared_error(test_labels, untuned_pred), 3))

starting MF RMSE: 0.338


## 5. Tune with 4-fold cross-validation (10 min)

Use `GridSearchCV` with these **rehearsal** settings:

- `K = [2, 3]`
- `lambda_ = [0.5, 2.0]`
- `cv = 4`
- `scoring = 'neg_root_mean_squared_error'`
- `refit=True`

Then display the CV table and use `best_estimator_` for the local test score.

In [5]:
param_grid = {'K': [2, 3], 'lambda_': [0.5, 2.0]}
mf_search = GridSearchCV(
    estimator=BiasedMF(n_users=n_users, n_items=n_items, n_iters=3, random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=4,
    refit=True,
)
mf_search.fit(X_train, y_train)
cv_table = pd.DataFrame(mf_search.cv_results_)[
    ['param_K', 'param_lambda_', 'mean_test_score', 'rank_test_score']
].copy()
cv_table['validation_RMSE'] = -cv_table.pop('mean_test_score')
display(cv_table.sort_values('validation_RMSE').round(3))
print('selected settings:', mf_search.best_params_)
tuned_pred = mf_search.best_estimator_.predict(X_test)
print('tuned MF RMSE:', round(root_mean_squared_error(test_labels, tuned_pred), 3))

,param_K,param_lambda_,rank_test_score,validation_RMSE
0,2,0.5,1,0.316
2,3,0.5,2,0.331
1,2,2.0,3,0.365
3,3,2.0,4,0.368


selected settings: {'K': 2, 'lambda_': 0.5}
tuned MF RMSE: 0.295


## 6. Compare and explain (3 min)

| Model | Local RMSE |
| --- | ---: |
| Global mean | 0.939 |
| User mean | 0.972 |
| Item mean | 0.951 |
| Starting MF | 0.338 |
| Tuned MF | 0.295 |

1. The tuned MF has the lowest local RMSE. Among the baselines, the global mean is best.
2. Tuning improved the local RMSE from 0.338 to 0.295 (selected `K=2`, `lambda_=0.5`).
3. `test_labels` must stay unseen during model selection. Using them to choose `K` and `lambda_` would leak the final evaluation into training. CV gives each candidate a validation fold instead.
4. Check whether the user or item appeared in the training rows. The supplied model uses zero bias/factor for unseen IDs, so predictions fall back to the available global mean and bias terms; the mean baselines fall back to the global mean.

**Exam checklist:** read the data → encode IDs consistently → preserve order →
baselines → MF with CV → predict `X_test` in order → submit.
